# 📋 W4: 출결 관리 & 안내문 자동화
**hexa-4 — Week 4** | ⏱️ 60분 | 무단결석 자동감지 + 학부모 안내문

## Step 0: 환경 설정

In [ ]:
import subprocess
subprocess.run(['apt-get','-qq','-y','install','fonts-nanum'],capture_output=True)
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
_n=[f for f in fm.findSystemFonts() if 'Nanum' in f]
if _n: fm.fontManager.addfont(_n[0]); plt.rcParams['font.family']=fm.FontProperties(fname=_n[0]).get_name()
plt.rcParams['axes.unicode_minus']=False
!pip install -q google-generativeai pandas matplotlib
try:
    from google.colab import userdata; API_KEY=userdata.get('GEMINI_API_KEY')
except: API_KEY=input('GEMINI_API_KEY: ')
import google.generativeai as genai; genai.configure(api_key=API_KEY)
model=genai.GenerativeModel('gemini-2.0-flash'); print('✅ 준비 완료')


## Step 1: 출결 데이터 입력

In [ ]:
import pandas as pd, numpy as np
df=pd.DataFrame({'날짜':pd.date_range('2026-01-02',periods=20,freq='B').strftime('%Y-%m-%d').tolist()*1,
    '김민준':[1]*18+[0,0],'이서연':[1]*20,'박도현':[1]*15+[0]*5,'최지아':[1]*20,'정예준':[1]*10+[0]*10})
df.set_index('날짜',inplace=True)
attend_rate=(df.sum()/len(df)*100).round(1)
print('📊 출석률:')
print(attend_rate.to_string())


## Step 2: 결석 현황 시각화

In [ ]:
import matplotlib.pyplot as plt
colors=['#e74c3c' if r<80 else '#f39c12' if r<95 else '#2ecc71' for r in attend_rate]
plt.figure(figsize=(8,4))
plt.bar(attend_rate.index,attend_rate.values,color=colors)
plt.axhline(80,color='red',linestyle='--',label='경고선(80%)')
plt.title('학생별 출석률'); plt.ylabel('%'); plt.ylim(0,110); plt.legend()
plt.tight_layout(); plt.savefig('attendance.png',dpi=150,bbox_inches='tight'); plt.show()


## Step 3: 무단결석 학생 안내문 자동 생성

In [ ]:
at_risk=attend_rate[attend_rate<80].index.tolist()
print(f'⚠️ 관심 필요 학생: {at_risk}')
for student in at_risk:
    absences=len(df)-df[student].sum()
    p=f"{student} 학생 출석률 {attend_rate[student]}% ({absences}회 결석). 학부모께 보내는 정중한 안내문 150자 이내."
    msg=model.generate_content(p).text.strip()
    print(f'\n[{student}]\n{msg}')


## Step 4: 다운로드

In [ ]:
attend_rate.to_csv('attendance_report.csv',header=['출석률'],encoding='utf-8-sig')
from google.colab import files
files.download('attendance_report.csv'); files.download('attendance.png')
print('✅ 완료!')
